# MLP S1 — Case Study: Multi-Source Store Sales

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/ms2a-machine-learning-practice/challenges/mlp-s1-case-study.ipynb)

Five stores sell products from the same catalogue of 1,000 items. The goal
is to predict `number_sold` for the products of **StoreE**. StoreE's own
file has no sales column, so a model has to learn from the four other
stores, whose data is spread over three kinds of source:

| Section | Source | What it provides |
|---|---|---|
| 1 | one file per store: CSV, Excel, JSON | product features and `number_sold` |
| 2 | a REST API protected by a password | the `volume` of every product |
| 3 | a web page rendered by JavaScript | `rating` and `num_reviews` of every product |
| 4 | StoreE's CSV + the same API and page | the table to predict |

Each section collects one source, joins it to the table built so far on
`product_id`, and measures the same simple baseline again, so the effect of
each source on the error is visible. The model is deliberately plain: the
subject here is collection, not modelling.

This notebook is the worked example for the Session 1 lessons on files and
formats, APIs and web scraping. It runs top to bottom in Colab.

---

## 0. Setup

Colab already provides pandas, scikit-learn, requests, BeautifulSoup and
openpyxl (the engine pandas uses for `.xlsx`); Selenium is not included.
The install runs only in Colab. Outside Colab, install the same packages in
your environment before running the notebook.

In [1]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ["selenium", "openpyxl", "beautifulsoup4", "requests"]

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *PACKAGES], check=True)
print("Running in Colab:", IN_COLAB)

Running in Colab: False


In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

---

## 1. Files

Each store exports its data in its own way. The file extension says which
reader to call; only the first lines of the file say which options that
reader needs.

### Download

The site hosting the files answers **status 200 with its HTML home page**
for any path it does not know, so a mistyped file name still "succeeds".
`raise_for_status()` alone would not detect that, which is why the loop
also rejects an HTML response.

In [3]:
BASE_URL = ("https://www.raphaelcousin.com/modules/"
            "data-science-practice/module4/course/")
FILES = ["StoreA_data.csv", "StoreB_data.csv", "StoreC_data.xlsx",
         "StoreD_data.json", "StoreE_data.csv"]

for name in FILES:
    r = requests.get(BASE_URL + name, timeout=10)
    r.raise_for_status()
    if r.headers["Content-Type"].startswith("text/html"):
        raise ValueError(f"{name}: received an HTML page, not the file")
    with open(name, "wb") as f:
        f.write(r.content)
    print(f"{name:18} {len(r.content):>6} bytes")

StoreA_data.csv     12241 bytes
StoreB_data.csv     11604 bytes
StoreC_data.xlsx    16999 bytes
StoreD_data.json    38856 bytes
StoreE_data.csv     11058 bytes


`peek` prints the first raw lines of a text file with `repr`, which makes
separators, empty fields and line endings visible.

In [4]:
def peek(name, n=3):
    with open(name, encoding="utf-8") as f:
        for _ in range(n):
            print(repr(f.readline()))

### StoreA: a standard CSV

Comma-separated, one header line, `product_id` in the first column: the
defaults of `read_csv` fit. `index_col="product_id"` makes the product id
the row label, which every later join aligns on.

In [5]:
peek("StoreA_data.csv")
df_A = pd.read_csv("StoreA_data.csv", index_col="product_id")
df_A.head(3)

'product_id,store,weight,length,width,days_since_last_sale,days_in_stock,price,number_sold,last_updated\n'
'P0019,StoreA,8.71,12.81,58.68,190,651,17.23,214,2023-01-19\n'
'P0024,StoreA,3.42,41.28,38.92,115,317,10.63,188,2023-01-24\n'


,store,weight,length,width,days_since_last_sale,days_in_stock,price,number_sold,last_updated
product_id,,,,,,,,,
P0019,StoreA,8.71,12.81,58.68,190,651,17.23,214,2023-01-19
P0024,StoreA,3.42,41.28,38.92,115,317,10.63,188,2023-01-24
P0025,StoreA,8.05,24.54,92.99,337,123,21.49,176,2023-01-25


### StoreB: a CSV with a different layout

The raw lines differ from StoreA in four ways, and each needs one option:

- fields are separated by `;` (usual where `,` is the decimal mark):
  `sep=";"`;
- the first line is a row of empty fields, so the header is on line 1:
  `header=1`;
- every line ends with `;`, which pandas reads as an extra, empty column
  named `Unnamed: 10`;
- column names are upper case. Left as they are, `concat` would put
  `WEIGHT` and `weight` side by side as two half-empty columns.

In [6]:
peek("StoreB_data.csv")
df_B = pd.read_csv("StoreB_data.csv", sep=";", header=1,
                   index_col="PRODUCT_ID")
df_B = df_B.drop(columns="Unnamed: 10")
df_B.columns = df_B.columns.str.lower()
df_B.index.name = "product_id"
df_B.head(3)

';;;;;;;;;;\n'
'PRODUCT_ID;STORE;WEIGHT;LENGTH;WIDTH;DAYS_SINCE_LAST_SALE;DAYS_IN_STOCK;PRICE;NUMBER_SOLD;LAST_UPDATED;\n'
'P0006;StoreB;1.61;75.06;91.25;301.0;783;8.26;220;2023-01-06;\n'


,store,weight,length,width,days_since_last_sale,days_in_stock,price,number_sold,last_updated
product_id,,,,,,,,,
P0006,StoreB,1.61,75.06,91.25,301.0,783,8.26,220,2023-01-06
P0014,StoreB,1.22,14.35,73.24,272.0,657,6.84,71,2023-01-14
P0016,StoreB,8.41,77.62,65.86,255.0,866,16.70,119,2023-01-16


### StoreC: an Excel workbook with two sheets

`read_excel` reads only the first sheet by default. `sheet_name=None`
returns every sheet in a dict, which shows how this workbook is organised.

In [7]:
sheets = pd.read_excel("StoreC_data.xlsx", sheet_name=None)
for name, sheet in sheets.items():
    print(name, sheet.shape, list(sheet.columns))

Sheet1 (190, 2) ['product_id', 'number_sold']
Sheet2 (190, 8) ['product id', 'store', 'weight', 'length', 'width', 'days_since_last_sale', 'days_in_stock', 'price']


`Sheet1` holds the sales and `Sheet2` the product features, with the key
spelled `product id` (a space instead of an underscore). The two sheets are
joined on that key. A join matches rows by value, so it does not depend on
both sheets listing the products in the same order; `validate="one_to_one"`
raises if a product appears twice in either sheet.

In [8]:
sales = sheets["Sheet1"].set_index("product_id")
features = sheets["Sheet2"].set_index("product id")
df_C = features.join(sales, how="inner", validate="one_to_one")
df_C.index.name = "product_id"
assert len(df_C) == len(sales) == len(features)
df_C.head(3)

,store,weight,length,width,days_since_last_sale,days_in_stock,price,number_sold
product_id,,,,,,,,
P0003,StoreC,7.00,11.77,98.21,120,513,17.85,28
P0007,StoreC,5.81,38.56,94.65,254,753,11.94,26
P0008,StoreC,6.11,33.17,96.21,121,540,7.24,34


### StoreD: JSON records

The file is a list of objects, one per product: the `records` orientation.
A file shaped `{"P0001": {...}, ...}` would need `orient="index"` instead,
and `{"weight": {"P0001": ...}, ...}` would need `orient="columns"`.
JSON has no index, so `product_id` stays an ordinary column until
`set_index`.

`last_updated` is stored here as a Unix timestamp in milliseconds, where
the CSV files store a `YYYY-MM-DD` string. It is converted to the same
string format so that the column has one representation after aggregation.

In [9]:
with open("StoreD_data.json", encoding="utf-8") as f:
    print(f.read(180))
df_D = pd.read_json("StoreD_data.json", orient="records")
df_D = df_D.set_index("product_id")
df_D["last_updated"] = (pd.to_datetime(df_D["last_updated"], unit="ms")
                        .dt.strftime("%Y-%m-%d"))
df_D.head(3)

[{"product_id":"P0001","store":"StoreD","weight":9.33,"length":73.05,"width":11.94,"days_since_last_sale":236,"days_in_stock":65,"price":12.0,"number_sold":40,"last_updated":167253


,store,weight,length,width,days_since_last_sale,days_in_stock,price,number_sold,last_updated
product_id,,,,,,,,,
P0001,StoreD,9.33,73.05,11.94,236,65,12.00,40,2023-01-01
P0011,StoreD,9.35,67.83,76.94,284,496,15.07,42,2023-01-11
P0015,StoreD,9.85,67.73,28.50,231,369,13.11,30,2023-01-15


### Aggregate the four stores

The four tables now share column names and the `product_id` index, so
`concat` stacks them row-wise. Columns are matched by name: a column that
one store lacks (StoreC has no `last_updated`) is filled with missing values
for that store's rows. Parsing `last_updated` with an explicit `format`
raises on any value that does not follow it.

In [10]:
data = pd.concat([df_A, df_B, df_C, df_D])
data["last_updated"] = pd.to_datetime(data["last_updated"],
                                      format="%Y-%m-%d")
print(data.shape)
data["store"].value_counts()

(796, 9)


store
StoreA    210
StoreD    206
StoreB    190
StoreC    190
Name: count, dtype: int64

### Check the aggregated table

Three questions before any model: is a product listed twice, where are the
values missing, and does every column have the type it should? A numeric
column read as text would show up here as `object` (`str` in pandas 3).

In [11]:
print("duplicated product_id:", data.index.duplicated().sum())
print("missing values per column:")
print(data.isna().sum()[lambda s: s > 0])
data.dtypes

duplicated product_id: 0
missing values per column:
weight                    6
length                    3
days_since_last_sale      3
last_updated            190
dtype: int64


store                           object
weight                         float64
length                         float64
width                          float64
days_since_last_sale           float64
days_in_stock                    int64
price                          float64
number_sold                      int64
last_updated            datetime64[ns]
dtype: object

No product is listed twice and the target `number_sold` is complete. The
gaps are 6 `weight`, 3 `length` and 3 `days_since_last_sale` values, plus
`last_updated` for all 190 StoreC rows.

`days_since_last_sale` is `float64` although it counts days. An integer
column cannot hold NaN, so a single missing value turns the whole column
into floats (StoreB's file already writes it as `301.0`). The assertions
below make the two properties the model relies on stop the notebook if they
ever fail.

In [12]:
assert not data.index.duplicated().any()
assert data["number_sold"].notna().all()

### A baseline to measure each source by

Missing values filled with -1, standardised features, a linear regression,
scored by the mean absolute error (MAE, in units sold) over 5-fold
cross-validation. The model stays the same for the whole notebook, so a
change in MAE comes from the data that was added.

`store` and `last_updated` are not used as features: StoreE never appears in
the training rows, so a store indicator cannot carry over to it, and a date
needs feature engineering before a linear model can use it.

In [13]:
TARGET = "number_sold"
NOT_FEATURES = [TARGET, "store", "last_updated"]


def make_model():
    return make_pipeline(StandardScaler(), LinearRegression())


def cv_mae(data, model):
    X = data.drop(columns=NOT_FEATURES).fillna(-1)
    folds = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, data[TARGET], cv=folds,
                             scoring="neg_mean_absolute_error")
    return -scores.mean()


mae = {"files": cv_mae(data, make_model())}
print(f"MAE, files only: {mae['files']:.2f}")

MAE, files only: 58.51


---

## 2. API

The volume of every product is served by a small REST API. Access takes two
calls: `/api/course/auth` returns a password, and the password is part of
the path of `/api/course/<password>/volumes`. Both return the same
envelope, `{"status": ..., "message": ..., "data": ...}`.

This API also answers a wrong path, a wrong password included, with the
site's HTML page and status 200 rather than an error status:

In [14]:
API = "https://www.raphaelcousin.com/api/course"
r = requests.get(f"{API}/not-the-password/volumes", timeout=10)
print(r.status_code, r.headers["Content-Type"])

200 text/html


`get_api` therefore refuses an HTML response and checks the envelope's
`status`, so a failed call stops at the call instead of surfacing later as
a confusing error. The password is kept in a variable and never printed:
a credential written into a notebook's code or output travels with every
copy of the notebook.

In [15]:
def get_api(url):
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    if r.headers["Content-Type"].startswith("text/html"):
        raise ValueError(f"{url} returned an HTML page, not JSON")
    body = r.json()
    if body["status"] != "success":
        raise ValueError(f"{url}: {body['message']}")
    print(body["message"])
    return body["data"]


password = get_api(f"{API}/auth")["password"]
volumes = get_api(f"{API}/{password}/volumes")
print(len(volumes), "products, e.g.", list(volumes.items())[:2])

Authentication successful
Volume data retrieved successfully
1000 products, e.g. [('P0001', 48469.09869), ('P0002', 11252.96382)]


`volumes` is a dict keyed by product id. `DataFrame.from_dict` with
`orient="index"` turns the keys into the row index; the default,
`orient="columns"`, would produce one row with 1,000 columns.

The API lists products in id order, while `data` is ordered store by store.
`join` aligns rows on the index labels, never on their position, so each
product receives its own volume. The left join keeps every row of `data`;
a product absent from the API would get a missing value, which the check
after the join counts.

In [16]:
df_volume = pd.DataFrame.from_dict(volumes, orient="index",
                                   columns=["volume"])
df_volume.index.name = "product_id"
data = data.join(df_volume, how="left", validate="one_to_one")
print("rows without a volume:", data["volume"].isna().sum())
mae["files + API"] = cv_mae(data, make_model())
print(f"MAE, files + API: {mae['files + API']:.2f}")

rows without a volume: 0


MAE, files + API: 57.74


---

## 3. Web scraping

Rating and number of reviews are published only as a table on a web page:
<https://www.raphaelcousin.com/module4/scrapable-data>. The page is built in
the browser by JavaScript, so `requests` receives the empty shell that the
browser starts from:

In [17]:
URL = "https://www.raphaelcousin.com/module4/scrapable-data"
shell = requests.get(URL, timeout=10).text
n_tables = len(BeautifulSoup(shell, "html.parser").find_all("table"))
print(len(shell), "characters,", n_tables, "tables")

1528 characters, 0 tables


There is no JSON request behind this page to call instead: the table's
values are compiled into the page's JavaScript bundle. A browser has to run
that script, and Selenium drives one: Chrome without a window (headless).

Colab has no Chrome. The next cell installs Google's official Debian package
of Chrome, and `apt-get` installs the system libraries it depends on. On the
first `webdriver.Chrome(...)`, Selenium (4.6 and later) downloads the
chromedriver that matches the installed Chrome. Outside Colab the cell does
nothing, and the Chrome installed on your machine is used.

In [18]:
if IN_COLAB:
    deb = "google-chrome-stable_current_amd64.deb"
    apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    subprocess.run(["wget", "-q",
                    f"https://dl.google.com/linux/direct/{deb}"],
                   check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", f"./{deb}"],
                   check=True, env=apt_env)

The three options are the ones a container needs:

- `--headless=new`: no window, since there is no display;
- `--no-sandbox`: Colab runs as `root`, and Chrome refuses to start its
  sandbox as root;
- `--disable-dev-shm-usage`: containers have a small `/dev/shm`; Chrome
  writes to `/tmp` instead.

`WebDriverWait` waits until the first table row exists, up to 30 seconds,
rather than sleeping for a fixed time that is either too long or too short.
`finally` closes the browser even if the wait fails.

In [19]:
options = webdriver.ChromeOptions()
for arg in ("--headless=new", "--no-sandbox", "--disable-dev-shm-usage"):
    options.add_argument(arg)

driver = webdriver.Chrome(options=options)
try:
    driver.get(URL)
    WebDriverWait(driver, 30).until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "table tbody tr")))
    html = driver.page_source
finally:
    driver.quit()
print(len(html), "characters after rendering")

193518 characters after rendering


The rendered page holds two tables: the one for this case study and one
for an exercise. Taking `tables[0]` or `tables[1]` would depend on the page
layout, which can change without notice. Selecting the table by its header
row raises a `KeyError` if the headers change, instead of silently reading
the wrong table.

In [20]:
soup = BeautifulSoup(html, "html.parser")
tables = {tuple(th.text for th in t.find_all("th")): t
          for t in soup.find_all("table")}
print(list(tables))
table = tables[("Product ID", "Rating", "Number of Reviews",
                "Updated Timestamp")]

[('Product ID', 'Rating', 'Number of Reviews', 'Updated Timestamp'), ('Item Code', 'Customer Score', 'Total Reviews', 'Updated Timestamp')]


Each row is unpacked into exactly four cells, which fails if a column is
added or removed. Cell text is converted to numbers explicitly: HTML only
carries strings. `Updated Timestamp` is regenerated at random on every page
load, so it describes the page, not the product, and is not kept.

In [21]:
rows = []
for tr in table.find("tbody").find_all("tr"):
    cells = [td.text for td in tr.find_all("td")]
    product_id, rating, reviews, _updated = cells
    rows.append({"product_id": product_id, "rating": float(rating),
                 "num_reviews": int(reviews)})

df_reviews = pd.DataFrame(rows).set_index("product_id")
print(df_reviews.shape)
df_reviews.head(3)

(1000, 2)


,rating,num_reviews
product_id,,
P0001,1.0,897
P0002,3.0,209
P0003,3.0,125


In [22]:
data = data.join(df_reviews, how="left", validate="one_to_one")
print("rows without a rating:", data["rating"].isna().sum())
mae["files + API + scraping"] = cv_mae(data, make_model())
pd.Series(mae, name="MAE").round(2)

rows without a rating: 0


files                     58.51
files + API               57.74
files + API + scraping    56.45
Name: MAE, dtype: float64

For scale: the MAE of a model that predicts the median of `number_sold` for
every product, on the same folds, and the average sales per product in each
store.

In [23]:
median_model = DummyRegressor(strategy="median")
print(f"MAE, median for every product: {cv_mae(data, median_model):.2f}")
data.groupby("store")[TARGET].mean().round(1)

MAE, median for every product: 60.03


store
StoreA    170.7
StoreB    126.4
StoreC     24.4
StoreD     42.0
Name: number_sold, dtype: float64

Each source lowers the error: 58.51 with the files alone, 57.74 with the
volume from the API, 56.45 with the scraped ratings and reviews. The gains
are modest, and so is the margin over the median predictor (60.03). Most of
the variation in `number_sold` lies between stores (171 units per product on
average at StoreA, 24 at StoreC), and a model trained on four stores cannot
learn the level of a fifth. More product-level sources do not remove that
limit; information about the stores themselves would.

---

## 4. Predict StoreE

StoreE's file has StoreA's layout without `number_sold`. Its extra columns
come from the same API and page, joined the same way. `X.columns` fixes the
feature order to the one used for training, and the model is refit on all
four stores.

In [24]:
df_E = pd.read_csv("StoreE_data.csv", index_col="product_id")
df_E = df_E.join(df_volume, how="left").join(df_reviews, how="left")
print(df_E.shape, "missing values:", int(df_E.isna().sum().sum()))

(204, 11) missing values: 0


In [25]:
X = data.drop(columns=NOT_FEATURES).fillna(-1)
model = make_model().fit(X, data[TARGET])

X_E = df_E[X.columns].fillna(-1)
pred_E = pd.Series(model.predict(X_E), index=df_E.index, name=TARGET)
pred_E.describe().round(1)

count    204.0
mean      95.9
std       36.6
min       31.4
25%       67.9
50%       92.8
75%      117.0
max      215.7
Name: number_sold, dtype: float64

---

## Summary

- The options of a file reader (`sep`, `header`, `sheet_name`, `orient`)
  are read off the raw file, not guessed from its extension.
- A status 200 does not prove the content is the one requested; check what
  came back.
- Sources are combined by joining on a key (`product_id`), never by row
  position, and each join is followed by a count of the rows it left
  unmatched.
- Duplicates, missing values and dtypes are checked once the sources are
  aggregated, before any model.
- Scraping a rendered page needs a browser, which is slower and more
  fragile than a file or an API: locate the data by content (the table
  headers), not by position.

The Session 1 challenge, *MLP S1 — Multi-Source Store Sales*, applies the
same steps to another set of stores.